# RoadSafe Analytics — Data Cleaning

This notebook cleans the collision, vehicle, and casualty datasets. Raw files are preserved and cleaned Parquet files are saved in `data/processed/`.

In [ ]:
from pathlib import Path
import re
import pandas as pd

# This works when the notebook is opened from the project root or notebooks folder.
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

print('Project:', project_root)
print('Raw folder exists:', raw_dir.exists())
print('Processed folder:', processed_dir)

In [ ]:
raw_files = sorted(raw_dir.glob('*.csv'))
if len(raw_files) == 0:
    raise FileNotFoundError(f'No CSV files found in {raw_dir}')

print('Raw CSV files found:')
for file in raw_files:
    print(' -', file.name)

In [ ]:
def locate(keyword):
    matches = [file for file in raw_files if keyword in file.name.lower()]
    if not matches:
        raise FileNotFoundError(f'Could not find a CSV containing: {keyword}')
    return matches[0]

collision_file = locate('collision')
vehicle_file = locate('vehicle')
casualty_file = locate('casual')

print(collision_file.name)
print(vehicle_file.name)
print(casualty_file.name)

In [ ]:
def clean_data(df):
    before_rows = len(df)

    # Standardise column names.
    df.columns = [
        re.sub(r'[^a-z0-9]+', '_', str(column).strip().lower()).strip('_')
        for column in df.columns
    ]

    # Convert common placeholders to missing values.
    missing_values = ['', ' ', 'NA', 'N/A', 'na', 'n/a', 'NULL', 'null', '-', '?']
    df = df.replace(missing_values, pd.NA)

    # Remove empty rows, empty columns, and exact duplicate rows.
    df = df.dropna(axis=0, how='all')
    df = df.dropna(axis=1, how='all')
    df = df.drop_duplicates().reset_index(drop=True)

    # Remove unnecessary spaces in text columns.
    for column in df.select_dtypes(include=['object', 'string']).columns:
        df[column] = df[column].astype('string').str.strip()

    return df, before_rows, len(df)

In [ ]:
datasets = {
    'collisions_clean.parquet': collision_file,
    'vehicles_clean.parquet': vehicle_file,
    'casualties_clean.parquet': casualty_file,
}

summary = []
for output_name, input_file in datasets.items():
    original = pd.read_csv(input_file, low_memory=False)
    cleaned, rows_before, rows_after = clean_data(original)
    output_file = processed_dir / output_name
    cleaned.to_parquet(output_file, index=False)
    summary.append({
        'dataset': output_name,
        'rows_before': rows_before,
        'rows_after': rows_after,
        'columns_after': len(cleaned.columns),
        'missing_values_after': int(cleaned.isna().sum().sum()),
    })
    print('Saved:', output_file)

summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
print('\nFinal processed files:')
for file in sorted(processed_dir.glob('*_clean.parquet')):
    print(' -', file.name)

print('\nCleaning completed successfully.')